# UTCI Sulawesi 1996–2025 — STEP 1: QC Data Mentah

Notebook ini **belum menghitung daily maximum UTCI**. Fokus tahap pertama hanya memastikan data mentah aman.

Folder input:

`D:\MERLI\data\UTCI`

Folder output QC:

`D:\MERLI\data\UTCI_Hasil`

Berdasarkan contoh file yang diperiksa:
- variabel: `utci`
- dimensi per file: `24 × 57 × 45`
- satu file = satu hari
- 24 timestep = hourly
- koordinat: `time`, `lat`, `lon`
- riwayat NetCDF mencatat `-addc,273.15`, sehingga nilai file tersimpan dalam **Kelvin**
- konversi UTCI ke °C untuk QC: `UTCI_C = UTCI_K - 273.15`

## Target tahap ini

1. Memastikan semua file 1996–2025 ada;
2. memastikan satu hari berisi 24 jam;
3. memastikan selang waktu 1 jam;
4. memastikan grid semua file sama;
5. memastikan tidak ada tanggal/jam duplikat;
6. memastikan tidak ada hari hilang;
7. menyimpan tabel QC ke CSV.

**Jangan lanjut ke daily maximum sebelum QC ini lulus.**

In [1]:
from pathlib import Path
import gc
import calendar
import numpy as np
import pandas as pd
import xarray as xr

print("Python/xarray siap.")
print("xarray:", xr.__version__)
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

Python/xarray siap.
xarray: 2025.10.1
pandas: 2.3.3
numpy : 2.3.5


In [2]:
# =========================================================
# 1. KONFIGURASI
# =========================================================

DATA_DIR = Path(r"D:\MERLI\data\UTCI")
OUTPUT_DIR = Path(r"D:\MERLI\data\UTCI_Hasil")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

START_YEAR = 1996
END_YEAR = 2025

QC_FILE = OUTPUT_DIR / "01_qc_utci_raw_files_1996_2025.csv"
QC_DAILY_FILE = OUTPUT_DIR / "02_qc_utci_daily_dates_1996_2025.csv"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder UTCI tidak ditemukan:\n{DATA_DIR}"
    )

print("INPUT :", DATA_DIR)
print("OUTPUT:", OUTPUT_DIR)

INPUT : D:\MERLI\data\UTCI
OUTPUT: D:\MERLI\data\UTCI_Hasil


## 2. Cari file UTCI

Pencarian memakai `glob("*.nc")`, **bukan `rglob`**, supaya output atau file lain di subfolder tidak ikut terbaca.

In [3]:
# =========================================================
# 2. CARI FILE NETCDF
# =========================================================

UTCI_DATA_DIR = DATA_DIR / "1996-2025"

if not UTCI_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder UTCI 1996-2025 tidak ditemukan:\n{UTCI_DATA_DIR}"
    )


nc_files = sorted(
    UTCI_DATA_DIR.glob("*.nc")
)

print(
    "Folder data UTCI:",
    UTCI_DATA_DIR
)

print(
    "Jumlah file UTCI ditemukan:",
    len(nc_files)
)


if len(nc_files) == 0:
    raise FileNotFoundError(
        f"Tidak ada file .nc di:\n{UTCI_DATA_DIR}"
    )


print("\n10 file pertama:")

for path in nc_files[:10]:
    print(
        " -",
        path.name
    )


print("\n5 file terakhir:")

for path in nc_files[-5:]:
    print(
        " -",
        path.name
    )


expected_days = pd.date_range(
    f"{START_YEAR}-01-01",
    f"{END_YEAR}-12-31",
    freq="D"
)


print(
    "\nJumlah hari ideal 1996–2025:",
    len(expected_days)
)


if len(nc_files) != len(expected_days):

    print(
        "\nPERINGATAN:"
    )

    print(
        "Jumlah file tidak sama dengan jumlah hari ideal."
    )

    print(
        "Belum tentu fatal. "
        "QC tanggal berikutnya akan mencari "
        "hari hilang atau duplikat."
    )

else:

    print(
        "\nJumlah file sesuai dengan jumlah hari ideal."
    )

Folder data UTCI: D:\MERLI\data\UTCI\1996-2025
Jumlah file UTCI ditemukan: 10958

10 file pertama:
 - ECMWF_utci_19960101_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960102_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960103_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960104_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960105_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960106_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960107_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960108_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960109_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_19960110_v1.1_con.area-subset.7.127.-7.116.nc

5 file terakhir:
 - ECMWF_utci_20251227_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_20251228_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_20251229_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_20251230_v1.1_con.area-subset.7.127.-7.116.nc
 - ECMWF_utci_20251231_v1.1_con.area-subset

## 3. Inspeksi satu file contoh

Cell ini memeriksa struktur file pertama sebelum looping ribuan file.

In [8]:
# =========================================================
# 2. CARI FILE NETCDF - RECURSIVE
# =========================================================

from pathlib import Path

UTCI_DATA_DIR = Path(
    r"D:\MERLI\data\UTCI\1996-2025"
)

if not UTCI_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder tidak ditemukan:\n{UTCI_DATA_DIR}"
    )


# Cari semua .nc / .NC sampai ke subfolder
nc_files = sorted(
    [
        p
        for p in UTCI_DATA_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower() == ".nc"
    ]
)


print(
    "Folder data UTCI:",
    UTCI_DATA_DIR
)

print(
    "Jumlah file UTCI ditemukan:",
    len(nc_files)
)


# =========================================================
# CEK KALAU MASIH KOSONG
# =========================================================

if len(nc_files) == 0:

    print(
        "\nIsi folder yang ditemukan:"
    )

    for p in UTCI_DATA_DIR.iterdir():
        print(
            " -",
            p.name
        )

    raise FileNotFoundError(
        "Masih tidak menemukan file NetCDF. "
        "Cek apakah file UTCI benar-benar berekstensi .nc."
    )


# =========================================================
# TAMPILKAN CONTOH FILE
# =========================================================

print(
    "\n10 file pertama:"
)

for path in nc_files[:10]:
    print(
        " -",
        path
    )


print(
    "\n5 file terakhir:"
)

for path in nc_files[-5:]:
    print(
        " -",
        path
    )


# =========================================================
# JUMLAH HARI IDEAL
# =========================================================

expected_days = pd.date_range(
    f"{START_YEAR}-01-01",
    f"{END_YEAR}-12-31",
    freq="D"
)


print(
    "\nJumlah hari ideal 1996–2025:",
    len(expected_days)
)


if len(nc_files) != len(expected_days):

    print(
        "\nPERINGATAN:"
    )

    print(
        f"File ditemukan : {len(nc_files)}"
    )

    print(
        f"Hari seharusnya: {len(expected_days)}"
    )

    print(
        "Belum tentu masalah. "
        "QC berikutnya akan mengecek tanggal hilang/duplikat."
    )

else:

    print(
        "\nJumlah file sesuai dengan jumlah hari ideal."
    )

Folder data UTCI: D:\MERLI\data\UTCI\1996-2025
Jumlah file UTCI ditemukan: 10958

10 file pertama:
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960101_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960102_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960103_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960104_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960105_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960106_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960107_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960108_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960109_v1.1_con.area-subset.7.127.-7.116.nc
 - D:\MERLI\data\UTCI\1996-2025\ECMWF_utci_19960110_v1.1_con.area-subset.7.127.-7.116.nc

5 file ter

## 4. QC semua file

Loop dilakukan **satu file per satu file**, lalu file langsung ditutup. Jadi RAM tidak menumpuk walaupun file berjumlah sekitar 10 ribu.

Output progres hanya dicetak setiap 500 file supaya notebook tidak penuh sampah output.

In [9]:
# =========================================================
# 4. FUNGSI QC SATU FILE
# =========================================================

def qc_one_utci_file(path, ref_lat=None, ref_lon=None):
    with xr.open_dataset(
        path,
        decode_times=True,
        decode_timedelta=False
    ) as ds:

        if "utci" not in ds.data_vars:
            raise ValueError(
                f"{path.name}: variabel 'utci' tidak ditemukan."
            )

        da = ds["utci"]

        # Nama koordinat
        if "lat" in ds.coords:
            lat_name = "lat"
        elif "latitude" in ds.coords:
            lat_name = "latitude"
        else:
            raise ValueError(
                f"{path.name}: koordinat latitude tidak ditemukan."
            )

        if "lon" in ds.coords:
            lon_name = "lon"
        elif "longitude" in ds.coords:
            lon_name = "longitude"
        else:
            raise ValueError(
                f"{path.name}: koordinat longitude tidak ditemukan."
            )

        if "time" not in ds.coords:
            raise ValueError(
                f"{path.name}: koordinat time tidak ditemukan."
            )

        lat = np.asarray(
            ds[lat_name].values,
            dtype=float
        )

        lon = np.asarray(
            ds[lon_name].values,
            dtype=float
        )

        times = pd.DatetimeIndex(
            ds["time"].values
        )

        # Grid consistency
        grid_ok = True

        if ref_lat is not None:
            grid_ok = (
                lat.shape == ref_lat.shape
                and lon.shape == ref_lon.shape
                and np.allclose(lat, ref_lat)
                and np.allclose(lon, ref_lon)
            )

        # Time interval
        if len(times) > 1:
            deltas = np.diff(
                times.values.astype("datetime64[ns]")
            )

            one_hour = np.timedelta64(1, "h")

            hourly_ok = bool(
                np.all(deltas == one_hour)
            )
        else:
            hourly_ok = False

        # Semua timestep harus berada dalam tanggal yang sama
        unique_dates = pd.DatetimeIndex(
            times.normalize().unique()
        )

        one_day_file = (
            len(unique_dates) == 1
        )

        # Range raw
        raw_min = float(
            da.min(skipna=True).values
        )

        raw_max = float(
            da.max(skipna=True).values
        )

        # Data contoh menunjukkan file tersimpan dalam Kelvin
        min_c = raw_min - 273.15
        max_c = raw_max - 273.15

        nan_count = int(
            da.isnull().sum().values
        )

        result = {
            "file": path.name,
            "path": str(path),
            "date": unique_dates[0] if one_day_file else pd.NaT,
            "time_start": times.min(),
            "time_end": times.max(),
            "n_time": len(times),
            "n_unique_dates": len(unique_dates),
            "hourly_interval_ok": hourly_ok,
            "one_day_file_ok": one_day_file,
            "grid_ok": grid_ok,
            "n_lat": len(lat),
            "n_lon": len(lon),
            "raw_min": raw_min,
            "raw_max": raw_max,
            "min_C": min_c,
            "max_C": max_c,
            "nan_count": nan_count,
        }

    return result, lat, lon, times

In [10]:
# =========================================================
# 5. LOOP QC SEMUA FILE
# =========================================================

records = []
all_hourly_times = []

ref_lat = None
ref_lon = None

for i, path in enumerate(
    nc_files,
    start=1
):
    row, lat, lon, times = qc_one_utci_file(
        path,
        ref_lat=ref_lat,
        ref_lon=ref_lon
    )

    if ref_lat is None:
        ref_lat = lat.copy()
        ref_lon = lon.copy()

        # File pertama otomatis grid referensi
        row["grid_ok"] = True

    records.append(row)
    all_hourly_times.append(times.values)

    if (
        i <= 5
        or i % 500 == 0
        or i == len(nc_files)
    ):
        print(
            f"[{i:05d}/{len(nc_files):05d}] "
            f"{path.name} | "
            f"{row['time_start']} → {row['time_end']}"
        )

    gc.collect()

qc_df = pd.DataFrame(records)

qc_df.to_csv(
    QC_FILE,
    index=False
)

print("\nQC file tersimpan:")
print(QC_FILE)

[00001/10958] ECMWF_utci_19960101_v1.1_con.area-subset.7.127.-7.116.nc | 1996-01-01 00:00:00 → 1996-01-01 23:00:00
[00002/10958] ECMWF_utci_19960102_v1.1_con.area-subset.7.127.-7.116.nc | 1996-01-02 00:00:00 → 1996-01-02 23:00:00
[00003/10958] ECMWF_utci_19960103_v1.1_con.area-subset.7.127.-7.116.nc | 1996-01-03 00:00:00 → 1996-01-03 23:00:00
[00004/10958] ECMWF_utci_19960104_v1.1_con.area-subset.7.127.-7.116.nc | 1996-01-04 00:00:00 → 1996-01-04 23:00:00
[00005/10958] ECMWF_utci_19960105_v1.1_con.area-subset.7.127.-7.116.nc | 1996-01-05 00:00:00 → 1996-01-05 23:00:00
[00500/10958] ECMWF_utci_19970514_v1.1_con.area-subset.7.127.-7.116.nc | 1997-05-14 00:00:00 → 1997-05-14 23:00:00
[01000/10958] ECMWF_utci_19980926_v1.1_con.area-subset.7.127.-7.116.nc | 1998-09-26 00:00:00 → 1998-09-26 23:00:00
[01500/10958] ECMWF_utci_20000208_v1.1_con.area-subset.7.127.-7.116.nc | 2000-02-08 00:00:00 → 2000-02-08 23:00:00
[02000/10958] ECMWF_utci_20010622_v1.1_con.area-subset.7.127.-7.116.nc | 2001-06

## 5. Ringkasan QC file

Target ideal:
- `n_time = 24`
- `hourly_interval_ok = True`
- `one_day_file_ok = True`
- `grid_ok = True`
- `nan_count = 0` atau nilai yang dapat dijelaskan

In [11]:
# =========================================================
# 6. RINGKASAN QC
# =========================================================

print("========== QC UTCI FILE ==========")

print("Jumlah file:", len(qc_df))

print("\nJumlah timestep per file:")
print(
    qc_df["n_time"]
    .value_counts()
    .sort_index()
)

print("\nHourly interval:")
print(
    qc_df["hourly_interval_ok"]
    .value_counts(dropna=False)
)

print("\nSatu hari per file:")
print(
    qc_df["one_day_file_ok"]
    .value_counts(dropna=False)
)

print("\nGrid konsisten:")
print(
    qc_df["grid_ok"]
    .value_counts(dropna=False)
)

print("\nNaN:")
print("Total NaN seluruh file:", int(qc_df["nan_count"].sum()))
print("File dengan NaN:", int((qc_df["nan_count"] > 0).sum()))

print("\nRange UTCI raw seluruh periode:")
print("Minimum:", qc_df["raw_min"].min())
print("Maximum:", qc_df["raw_max"].max())

print("\nRange UTCI Celsius seluruh periode:")
print("Minimum:", qc_df["min_C"].min(), "°C")
print("Maximum:", qc_df["max_C"].max(), "°C")

print("\nRentang tanggal file:")
print("Awal :", qc_df["date"].min())
print("Akhir:", qc_df["date"].max())

========== QC UTCI FILE ==========
Jumlah file: 10958

Jumlah timestep per file:
n_time
24    10958
Name: count, dtype: int64

Hourly interval:
hourly_interval_ok
True    10958
Name: count, dtype: int64

Satu hari per file:
one_day_file_ok
True    10958
Name: count, dtype: int64

Grid konsisten:
grid_ok
True    10958
Name: count, dtype: int64

NaN:
Total NaN seluruh file: 73
File dengan NaN: 7

Range UTCI raw seluruh periode:
Minimum: 275.93890380859375
Maximum: 317.6231689453125

Range UTCI Celsius seluruh periode:
Minimum: 2.7889038085937727 °C
Maximum: 44.47316894531252 °C

Rentang tanggal file:
Awal : 1996-01-01 00:00:00
Akhir: 2025-12-31 00:00:00


## 6. QC kalender harian

Bagian ini memeriksa apakah periode 1996–2025 lengkap dari 1 Januari 1996 sampai 31 Desember 2025.

In [12]:
# =========================================================
# 7. QC TANGGAL HARIAN
# =========================================================

file_dates = pd.DatetimeIndex(
    pd.to_datetime(
        qc_df["date"],
        errors="coerce"
    ).dropna()
).normalize()

duplicate_dates = file_dates[
    file_dates.duplicated()
]

missing_dates = expected_days.difference(
    file_dates
)

extra_dates = file_dates.difference(
    expected_days
)

print("========== QC KALENDER HARIAN ==========")

print("Jumlah tanggal file :", len(file_dates))
print("Jumlah hari harapan :", len(expected_days))
print("Tanggal duplikat    :", len(duplicate_dates))
print("Tanggal hilang      :", len(missing_dates))
print("Tanggal ekstra      :", len(extra_dates))

if len(duplicate_dates):
    print("\nContoh tanggal duplikat:")
    print(duplicate_dates[:20].tolist())

if len(missing_dates):
    print("\nContoh tanggal hilang:")
    print(missing_dates[:20].tolist())

if len(extra_dates):
    print("\nContoh tanggal ekstra:")
    print(extra_dates[:20].tolist())

# Tabel jumlah file per tahun
actual_year_counts = (
    pd.Series(file_dates.year)
    .value_counts()
    .sort_index()
)

year_rows = []

for year in range(
    START_YEAR,
    END_YEAR + 1
):
    expected_n = (
        366
        if calendar.isleap(year)
        else 365
    )

    actual_n = int(
        actual_year_counts.get(
            year,
            0
        )
    )

    year_rows.append(
        {
            "year": year,
            "actual_days": actual_n,
            "expected_days": expected_n,
            "OK": actual_n == expected_n,
        }
    )

daily_qc_df = pd.DataFrame(
    year_rows
)

print("\n========== QC PER TAHUN ==========")
print(
    daily_qc_df.to_string(
        index=False
    )
)

daily_qc_df.to_csv(
    QC_DAILY_FILE,
    index=False
)

print("\nQC kalender tersimpan:")
print(QC_DAILY_FILE)

========== QC KALENDER HARIAN ==========
Jumlah tanggal file : 10958
Jumlah hari harapan : 10958
Tanggal duplikat    : 0
Tanggal hilang      : 0
Tanggal ekstra      : 0

========== QC PER TAHUN ==========
 year  actual_days  expected_days   OK
 1996          366            366 True
 1997          365            365 True
 1998          365            365 True
 1999          365            365 True
 2000          366            366 True
 2001          365            365 True
 2002          365            365 True
 2003          365            365 True
 2004          366            366 True
 2005          365            365 True
 2006          365            365 True
 2007          365            365 True
 2008          366            366 True
 2009          365            365 True
 2010          365            365 True
 2011          365            365 True
 2012          366            366 True
 2013          365            365 True
 2014          365            365 True
 2015          

## 7. QC seluruh timestamp hourly

Dengan 10.958 hari × 24 jam, jumlah timestamp ideal adalah:

`262.992 timestep hourly`

Bagian ini mengecek duplikat dan kelengkapan hourly secara keseluruhan.

In [13]:
# =========================================================
# 8. QC HOURLY GLOBAL
# =========================================================

hourly_values = np.concatenate(
    all_hourly_times
)

hourly_index = pd.DatetimeIndex(
    hourly_values
).sort_values()

expected_hourly = pd.date_range(
    f"{START_YEAR}-01-01 00:00:00",
    f"{END_YEAR}-12-31 23:00:00",
    freq="h"
)

hourly_dup = hourly_index[
    hourly_index.duplicated()
]

hourly_missing = expected_hourly.difference(
    hourly_index
)

hourly_extra = hourly_index.difference(
    expected_hourly
)

print("========== QC HOURLY GLOBAL ==========")

print("Timestep aktual  :", len(hourly_index))
print("Timestep harapan :", len(expected_hourly))
print("Duplikat hourly  :", len(hourly_dup))
print("Missing hourly   :", len(hourly_missing))
print("Extra hourly     :", len(hourly_extra))

if len(hourly_dup):
    print("\nContoh duplicate hourly:")
    print(hourly_dup[:20].tolist())

if len(hourly_missing):
    print("\nContoh missing hourly:")
    print(hourly_missing[:20].tolist())

if len(hourly_extra):
    print("\nContoh extra hourly:")
    print(hourly_extra[:20].tolist())

========== QC HOURLY GLOBAL ==========
Timestep aktual  : 262992
Timestep harapan : 262992
Duplikat hourly  : 0
Missing hourly   : 0
Extra hourly     : 0


## 8. Status akhir STEP 1

Notebook akan menyatakan **LULUS** hanya jika:
- jumlah file = jumlah hari 1996–2025;
- semua file punya 24 timestep;
- interval waktu semua file 1 jam;
- satu file hanya berisi satu tanggal;
- grid seluruh file konsisten;
- tidak ada tanggal hilang/duplikat;
- tidak ada timestamp hourly hilang/duplikat.

NaN dilaporkan terpisah dan tidak otomatis membuat proses berhenti karena perlu dilihat konteks spasialnya.

In [14]:
# =========================================================
# 9. STATUS AKHIR
# =========================================================

checks = {
    "jumlah_file": len(qc_df) == len(expected_days),
    "24_timestep_per_file": bool((qc_df["n_time"] == 24).all()),
    "interval_1_jam": bool(qc_df["hourly_interval_ok"].all()),
    "satu_hari_per_file": bool(qc_df["one_day_file_ok"].all()),
    "grid_konsisten": bool(qc_df["grid_ok"].all()),
    "tanggal_duplikat_0": len(duplicate_dates) == 0,
    "tanggal_hilang_0": len(missing_dates) == 0,
    "tanggal_ekstra_0": len(extra_dates) == 0,
    "hourly_duplikat_0": len(hourly_dup) == 0,
    "hourly_missing_0": len(hourly_missing) == 0,
    "hourly_extra_0": len(hourly_extra) == 0,
}

print("========== STATUS STEP 1 ==========")

for name, status in checks.items():
    print(
        f"{name:24s}:",
        "LULUS" if status else "GAGAL"
    )

all_ok = all(
    checks.values()
)

print("\nHASIL AKHIR:")

if all_ok:
    print("QC UTCI RAW: LULUS")
    print(
        "Data siap masuk STEP 2: "
        "konversi K → °C + Daily Maximum UTCI + mask Sulawesi."
    )
else:
    print("QC UTCI RAW: BELUM LULUS")
    print(
        "Jangan lanjut ke daily maximum sebelum masalah QC diperiksa."
    )

========== STATUS STEP 1 ==========
jumlah_file             : LULUS
24_timestep_per_file    : LULUS
interval_1_jam          : LULUS
satu_hari_per_file      : LULUS
grid_konsisten          : LULUS
tanggal_duplikat_0      : LULUS
tanggal_hilang_0        : LULUS
tanggal_ekstra_0        : LULUS
hourly_duplikat_0       : LULUS
hourly_missing_0        : LULUS
hourly_extra_0          : LULUS

HASIL AKHIR:
QC UTCI RAW: LULUS
Data siap masuk STEP 2: konversi K → °C + Daily Maximum UTCI + mask Sulawesi.


In [15]:
# =========================================================
# 10. BERSIHKAN MEMORI
# =========================================================

del all_hourly_times
del hourly_values

gc.collect()

print("Memori sementara dibersihkan.")

Memori sementara dibersihkan.
